# 3.1b Run Inference Pipeline (job step)

Used by the **SWSAcademy - 03 Model Deployment** job. It does automatically what demo **3.1a**
asks students to do in the UI:

1. Create (or update) a serverless Lakeflow pipeline from **Pipeline/3.1b Demo - Inference Pipeline**
   with the three `mlpipeline.*` configuration values.
2. Run it and wait until it finishes, failing the job task if the pipeline fails.

Run **3.1a Demo - Pipeline Deployment** first: it registers the model this pipeline loads.
The notebook works out every path from its own location, so it works wherever the course folder was imported.

In [ ]:
import re
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
pipeline_notebook = f"{here.rsplit('/', 1)[0]}/Pipeline/3.1b Demo - Inference Pipeline"

# Same learner schema rule as the admin setup: email prefix, non-alphanumerics replaced by "_"
user = spark.sql("SELECT current_user() AS u").first()["u"]
schema = re.sub(r"[^A-Za-z0-9_]", "_", user.split("@")[0]).lower()
catalog = "swsacademy"
pipeline_name = f"SWSAcademy - 03 Inference Pipeline ({schema})"

spec = {
    "name": pipeline_name,
    "serverless": True,
    "catalog": catalog,
    "schema": schema,
    "development": True,
    "channel": "CURRENT",
    "tags": {"course": "swsacademy", "purpose": "instructor-demo"},
    "libraries": [{"notebook": {"path": pipeline_notebook}}],
    "configuration": {
        "mlpipeline.bronze_dataset_path": "/Volumes/swsacademy_telco/v01/telco/telco-customer-churn-missing.csv",
        "mlpipeline.model_name": f"{catalog}.{schema}.model_3_1a_demo",
        "mlpipeline.model_alias": "pipeline",
    },
}
print(f"Pipeline:      {pipeline_name}")
print(f"Source:        {pipeline_notebook}")
print(f"Target schema: {catalog}.{schema}")

In [ ]:
existing = [p for p in w.pipelines.list_pipelines(filter=f"name LIKE '{pipeline_name}'")]
if existing:
    pipeline_id = existing[0].pipeline_id
    w.api_client.do("PUT", f"/api/2.0/pipelines/{pipeline_id}", body=dict(spec, id=pipeline_id))
    print(f"Updated pipeline {pipeline_id}")
else:
    pipeline_id = w.api_client.do("POST", "/api/2.0/pipelines", body=spec)["pipeline_id"]
    print(f"Created pipeline {pipeline_id}")

update_id = w.api_client.do("POST", f"/api/2.0/pipelines/{pipeline_id}/updates", body={"full_refresh": True})["update_id"]
print(f"Started update {update_id}; waiting for it to finish...")

state = None
while state not in ("COMPLETED", "FAILED", "CANCELED"):
    time.sleep(20)
    state = w.api_client.do("GET", f"/api/2.0/pipelines/{pipeline_id}/updates/{update_id}")["update"]["state"]
    print("  state:", state)

if state != "COMPLETED":
    raise RuntimeError(f"Pipeline update {update_id} finished as {state}. Open the pipeline '{pipeline_name}' to see the error.")

for table in ("raw_inputs", "features_input", "model_predictions"):
    print(f"{catalog}.{schema}.{table}: {spark.table(f'{catalog}.{schema}.{table}').count()} rows")